# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. Run image notebook 00 first, using the same QUICK_RUN setting.

Source: Original local package: 03_fastai/03_compare_image_models.ipynb

In [1]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '03_compare_image_models.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

Exercise folder: /content/HaiLong_Colab_Exercises/03_fastai
Run mode: FULL EXERCISE
Results: /content/drive/MyDrive/HaiLong_Colab_Results/03_fastai/results


## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Use the real photographic branch and retain its split, class counts, held-out results and model artefacts.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


# Lesson 3: compare image model configurations
Source task: https://github.com/fastai/course22/blob/master/03-which-image-models-are-best.ipynb
This is an independently written solution companion, not a copy of the instructor notebook or a course completion record. A controlled ResNet18/ResNet34 comparison uses the same held-out split. The small smoke run tests mechanics on synthetic images; its scores do not rank photographic model quality.


In [2]:
import os,json
from pathlib import Path
import numpy as np,pandas as pd,torch
import matplotlib.pyplot as plt
torch.manual_seed(42);torch.set_num_threads(1)
np.random.seed(42)
SMOKE=os.environ.get('COURSE_SMOKE','0')=='1'
results=Path('results');results.mkdir(exist_ok=True)

import time
from fastai.vision.all import *
path=Path('data/synthetic_image_smoke' if SMOKE else 'data/bird_forest')
if not path.exists(): path=results/('image_dataset_smoke' if SMOKE else 'image_dataset_full')
if not path.exists(): raise FileNotFoundError('Run notebook 00 to prepare and persist the image folders.')
settings=json.loads((results/'image_config.json').read_text())
assert settings['synthetic_smoke'] == SMOKE, 'Rerun notebook 00 with the same full/quick setting.'
files=get_image_files(path)
failed=set(verify_images(files));files=L(f for f in files if f not in failed)
splits=RandomSplitter(valid_pct=.2,seed=42)(files)
rows=[]
for name,architecture in [('resnet18',resnet18),('resnet34',resnet34)]:
    set_seed(42)
    block=DataBlock(blocks=(ImageBlock,CategoryBlock),get_items=lambda _:files,
       splitter=IndexSplitter(splits[1]),get_y=parent_label,item_tfms=Resize(64 if SMOKE else 192))
    dls=block.dataloaders(path,bs=8,num_workers=0)
    learn=vision_learner(dls,architecture,pretrained=not SMOKE,metrics=error_rate)
    start=time.perf_counter()
    with learn.no_bar(),learn.no_logging():learn.fine_tune(1 if SMOKE else 3)
    elapsed=time.perf_counter()-start
    probabilities,targets=learn.get_preds(dl=dls.valid)
    targets=targets.reshape(-1)
    rows.append({'model':name,'parameters':sum(p.numel() for p in learn.model.parameters()),
      'seconds':elapsed,'validation_accuracy':float((probabilities.argmax(1)==targets).float().mean()),
      'synthetic_smoke':SMOKE})
table=pd.DataFrame(rows);table.to_csv(results/'image_model_comparison.csv',index=False)
print(table.to_string(index=False))

Storage check (before_fit): artifacts 2.278 GB; configured budget 9.000 GB; additional estimate 0.194 GB; runtime free 75.757 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.278 GB; configured budget 9.000 GB; additional estimate 0.055 GB; runtime free 75.757 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.325 GB; configured budget 9.000 GB; additional estimate 0.062 GB; runtime free 75.709 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.379 GB; configured budget 9.000 GB; additional estimate 0.055 GB; runtime free 75.656 GB. Budget is not Drive quota.


Storage check (before_fit): artifacts 2.325 GB; configured budget 9.000 GB; additional estimate 0.596 GB; runtime free 75.707 GB. Budget is not Drive quota.


...

Storage check (checkpoint_write): artifacts 2.550 GB; configured budget 9.000 GB; additional estimate 0.357 GB; runtime free 75.394 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.899 GB; configured budget 9.000 GB; additional estimate 0.357 GB; runtime free 75.045 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.899 GB; configured budget 9.000 GB; additional estimate 0.357 GB; runtime free 75.045 GB. Budget is not Drive quota.


Storage check (checkpoint_write): artifacts 2.899 GB; configured budget 9.000 GB; additional estimate 0.096 GB; runtime free 74.695 GB. Budget is not Drive quota.


   model  parameters   seconds  validation_accuracy  synthetic_smoke
resnet18    11704896 76.448741                  1.0            False
resnet34    21813056 90.153904                  1.0            False


Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [3]:
finish_run(COLAB_CONTEXT)

Saved run settings. Save the notebook in Colab to keep its cell outputs.


In [4]:
# Added by the course runner: inspect the executed runtime, not coursework.
import os as _hailong_os, json as _hailong_json, platform as _hailong_platform
_hailong_evidence = {
    'python': _hailong_platform.python_version(),
    'quick_run': bool(globals().get('QUICK_RUN', False)),
    'course_smoke': _hailong_os.environ.get('COURSE_SMOKE', '0'),
    'keras_backend': _hailong_os.environ.get('KERAS_BACKEND'),
    'legacy_keras': _hailong_os.environ.get('TF_USE_LEGACY_KERAS'),
    'base_model_commit': getattr(getattr(globals().get('model'), 'config', None), '_commit_hash', None),
}
_hailong_context = globals().get('COLAB_CONTEXT')
if _hailong_context and _hailong_context['folder'].relative_to(_hailong_context['root']).as_posix().startswith(('03_fastai', '04_fastai_part2')):
    from course_persistence import collect_reuse_evidence as _hailong_collect
    _hailong_evidence['fastai_reuse_evidence'] = _hailong_context['run'].get('reuse_evidence') or _hailong_collect(_hailong_context)
print('__HAILONG_RUNTIME_EVIDENCE__' + _hailong_json.dumps(_hailong_evidence))


__HAILONG_RUNTIME_EVIDENCE__{"python": "3.12.13", "quick_run": false, "course_smoke": "0", "keras_backend": null, "legacy_keras": null, "base_model_commit": null, "fastai_reuse_evidence": {"contract_version": 1, "notebook": "03_fastai/03_compare_image_models.ipynb", "source_sha256": "61dfb3818f150b5c6a4f58cee9afdd73bfa02ccf92fb4a5cb685d88ebf40c4e2", "helper_sha256": {"03_fastai/convolution_workbook_exercise.py": "ecacb93aef6846c82d6a9eebc4809e1a6766410f04783ac132f247935af3a8fc", "03_fastai/fastai_utils.py": "a03e913e1c07f55aedd56ee6128096f3f952f4f8be22f83c4a22002589c4bd18", "03_fastai/wearable_objective_extension.py": "f9b0a8bf81a5bab8c10aee5e8795e9cef3df58e40925f7705e58301b18b3fbe9", "03_fastai/wearable_utils.py": "6de4ee45118d96e024bd4e0f3fd3552905b38a1ff01c0a5146578bb0d8263db9", "colab_runtime.py": "e11bedb13e1bd71a71c01cc49e558d05c89b057259fe52e64301d0c4d626a140", "course_persistence.py": "f14f7282d4b1e718b01858a61a2dca8b60f5135ba9971fc7923c50bf12cca3d2"}, "persistent_course_root":